In [ ]:
import torch
import torchvision
from torchvision.transforms import v2

transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 4

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
           

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# functions to show an image


def imshow(img):
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


# get some random training images
dataiter = iter(trainloader)
images, labels = next(dataiter)

# show images
imshow(torchvision.utils.make_grid(images))
# print labels
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(batch_size)))

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net()

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

# SGD = Stochastic Gradient Descent
# lr = learning rate
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [ ]:
num_epochs = 10

# Lists to store metrics for plotting later
train_losses = []    # average loss per epoch on training set
test_losses  = []    # average loss per epoch on test set
train_accs   = []    # accuracy per epoch on training set
test_accs    = []    # accuracy per epoch on test set

for epoch in range(num_epochs):

    net.train()  # set model to training mode
                 # (important for layers like Dropout, BatchNorm)

    running_loss   = 0.0
    correct_train  = 0
    total_train    = 0

    for i, data in enumerate(trainloader, 0):
        inputs, labels = data

        # Zero the gradients before each batch
        optimizer.zero_grad()

        # Forward pass: compute predictions
        outputs = net(inputs)

        # Compute how wrong the predictions are
        loss = criterion(outputs, labels)

        # Backward pass: compute gradients
        loss.backward()

        # Update weights using the gradients
        optimizer.step()

        # Track loss
        running_loss += loss.item()

        # Track accuracy on training set
        # outputs shape: (batch_size, 10) — one score per class
        # torch.max returns (max_value, index_of_max)
        # the index = the predicted class
        _, predicted = torch.max(outputs, 1)
        total_train   += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    # Average loss and accuracy over the whole training epoch
    epoch_train_loss = running_loss / len(trainloader)
    epoch_train_acc  = 100 * correct_train / total_train
    train_losses.append(epoch_train_loss)
    train_accs.append(epoch_train_acc)


    net.eval()  # set model to evaluation mode
                # (Dropout is disabled, BatchNorm uses running stats)

    test_loss    = 0.0
    correct_test = 0
    total_test   = 0

    with torch.no_grad():  # no gradients needed during evaluation
                           # saves memory and speeds up computation
        for data in testloader:
            inputs, labels = data
            outputs = net(inputs)

            loss = criterion(outputs, labels)
            test_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total_test   += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    epoch_test_loss = test_loss / len(testloader)
    epoch_test_acc  = 100 * correct_test / total_test
    test_losses.append(epoch_test_loss)
    test_accs.append(epoch_test_acc)

    print(f"Epoch {epoch+1:2d}/{num_epochs} | "
          f"Train Loss: {epoch_train_loss:.3f}, Train Acc: {epoch_train_acc:.1f}% | "
          f"Test  Loss: {epoch_test_loss:.3f},  Test Acc: {epoch_test_acc:.1f}%")

print("Training done!")

In [ ]:
epochs_range = range(1, num_epochs + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Both train and test loss on the same graph
ax1.plot(epochs_range, train_losses, label='Train Loss', marker='o')
ax1.plot(epochs_range, test_losses,  label='Test Loss',  marker='o')
ax1.set_title('Loss per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# Accuracy plot 
ax2.plot(epochs_range, train_accs, label='Train Accuracy', marker='o')
ax2.plot(epochs_range, test_accs,  label='Test Accuracy',  marker='o')
ax2.set_title('Accuracy per Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# What is a scheduler?
# During training, a fixed learning rate can be suboptimal:
#   - too high early on → unstable training
#   - too high later → overshoots the minimum
# A scheduler REDUCES the learning rate automatically over time.

# ExponentialLR multiplies lr by gamma after each epoch:
# epoch 1: lr = 0.001
# epoch 2: lr = 0.001 * 0.9  = 0.0009
# epoch 3: lr = 0.0009 * 0.9 = 0.00081
# ...

net_sched   = Net()
optimizer_s = optim.SGD(net_sched.parameters(), lr=0.001, momentum=0.9)
scheduler   = optim.lr_scheduler.ExponentialLR(optimizer_s, gamma=0.9)

train_losses_s, test_losses_s = [], []
train_accs_s,   test_accs_s   = [], []

for epoch in range(num_epochs):

    # Training phase
    net_sched.train()
    running_loss  = 0.0
    correct_train = 0
    total_train   = 0

    for inputs, labels in trainloader:
        optimizer_s.zero_grad()
        outputs = net_sched(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer_s.step()

        running_loss  += loss.item()
        _, predicted   = torch.max(outputs, 1)
        total_train   += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    # Step the scheduler ONCE per epoch (not per batch!)
    # This decays the learning rate by gamma
    scheduler.step()

    train_losses_s.append(running_loss / len(trainloader))
    train_accs_s.append(100 * correct_train / total_train)

    # ── Evaluation phase ─────────────────────────────────────────────────────
    net_sched.eval()
    test_loss    = 0.0
    correct_test = 0
    total_test   = 0

    with torch.no_grad():
        for inputs, labels in testloader:
            outputs   = net_sched(inputs)
            test_loss += criterion(outputs, labels).item()
            _, predicted = torch.max(outputs, 1)
            total_test   += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    test_losses_s.append(test_loss / len(testloader))
    test_accs_s.append(100 * correct_test / total_test)

    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch+1:2d} | LR: {current_lr:.6f} | "
          f"Train Acc: {train_accs_s[-1]:.1f}% | Test Acc: {test_accs_s[-1]:.1f}%")

In [ ]:
# TRANSFER LEARNING — ResNet18 on Oxford Pets

from torchvision.models import resnet18, ResNet18_Weights

# 1. Load pre-trained weights (trained on ImageNet — 1.2M images, 1000 classes)
weights   = ResNet18_Weights.IMAGENET1K_V1

# 2. Use the EXACT same preprocessing ResNet18 was trained with
#    (different from CIFAR10 normalization — critical step!)
transform_pets = weights.transforms()

# 3. Oxford Pets dataset — 37 breeds of cats and dogs
trainset_pets = torchvision.datasets.OxfordIIITPet(
    root='./data', split='trainval',
    transform=transform_pets, download=True)

testset_pets = torchvision.datasets.OxfordIIITPet(
    root='./data', split='test',
    transform=transform_pets, download=True)

trainloader_pets = torch.utils.data.DataLoader(
    trainset_pets, batch_size=32, shuffle=True, num_workers=2)
testloader_pets  = torch.utils.data.DataLoader(
    testset_pets,  batch_size=32, shuffle=False, num_workers=2)

# 4. Load the model with pre-trained weights
model = resnet18(weights=weights)

# 5. FREEZE all layers — we don't want to change what ResNet already learned
#    requires_grad=False means: "don't compute gradients for this parameter"
#    → optimizer.step() will NOT update these layers
for param in model.parameters():
    param.requires_grad = False

# 6. REPLACE the last layer (classifier)
#    ResNet18's original last layer: Linear(512 → 1000 ImageNet classes)
#    We need:                        Linear(512 → 37 pet breeds)
num_features = model.fc.in_features  # = 512
model.fc     = nn.Linear(num_features, 37)
# This new layer has requires_grad=True by default → will be trained

# 7. Optimizer only updates the NEW last layer's parameters
optimizer_tl = optim.SGD(model.fc.parameters(), lr=0.001, momentum=0.9)
criterion_tl = nn.CrossEntropyLoss()

# 8. Training loop — same structure as before
num_epochs_tl = 5  # fewer epochs needed — most weights are already good!

train_losses_tl, test_losses_tl = [], []
train_accs_tl,   test_accs_tl   = [], []

for epoch in range(num_epochs_tl):

    # ── Training ─────────────────────────────────────────────────────────────
    model.train()
    running_loss  = 0.0
    correct_train = 0
    total_train   = 0

    for inputs, labels in trainloader_pets:
        optimizer_tl.zero_grad()
        outputs = model(inputs)
        loss    = criterion_tl(outputs, labels)
        loss.backward()
        optimizer_tl.step()

        running_loss  += loss.item()
        _, predicted   = torch.max(outputs, 1)
        total_train   += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_losses_tl.append(running_loss / len(trainloader_pets))
    train_accs_tl.append(100 * correct_train / total_train)

    # ── Evaluation ───────────────────────────────────────────────────────────
    model.eval()
    test_loss    = 0.0
    correct_test = 0
    total_test   = 0

    with torch.no_grad():
        for inputs, labels in testloader_pets:
            outputs      = model(inputs)
            test_loss   += criterion_tl(outputs, labels).item()
            _, predicted = torch.max(outputs, 1)
            total_test   += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    test_losses_tl.append(test_loss / len(testloader_pets))
    test_accs_tl.append(100 * correct_test / total_test)

    print(f"Epoch {epoch+1}/{num_epochs_tl} | "
          f"Train Acc: {train_accs_tl[-1]:.1f}% | "
          f"Test  Acc: {test_accs_tl[-1]:.1f}%")

# 9. Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, num_epochs_tl+1), train_losses_tl, label='Train Loss', marker='o')
ax1.plot(range(1, num_epochs_tl+1), test_losses_tl,  label='Test Loss',  marker='o')
ax1.set_title('Transfer Learning — Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(True)

ax2.plot(range(1, num_epochs_tl+1), train_accs_tl, label='Train Accuracy', marker='o')
ax2.plot(range(1, num_epochs_tl+1), test_accs_tl,  label='Test Accuracy',  marker='o')
ax2.set_title('Transfer Learning — Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Load the same model and data as before
weights        = ResNet18_Weights.IMAGENET1K_V1
transform      = weights.transforms()

testset  = torchvision.datasets.OxfordIIITPet(
               root='./data', split='test',
               transform=transform, download=True)
testloader = torch.utils.data.DataLoader(
               testset, batch_size=32, shuffle=False)

# Load your already-trained transfer learning model
# (the one from the previous section)
model = resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, 37)
# If you saved it: model.load_state_dict(torch.load('model.pth'))
# Otherwise just use the model variable from the previous section

model.eval()  # always eval mode for visualization

# Helper: reverse the ImageNet normalization to display images normally
# transforms.Normalize subtracted mean and divided by std
# we need to undo that to show the original colors
def denormalize(tensor):
    """
    Reverses ImageNet normalization so the image looks normal when displayed.
    Input tensor shape: (C, H, W) with values roughly in [-2, 2]
    Output: numpy array (H, W, C) with values in [0, 1]
    """
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img  = tensor * std + mean          # undo normalization
    img  = img.clamp(0, 1)             # clip to valid range
    return img.permute(1, 2, 0).numpy()  # (C,H,W) → (H,W,C) for matplotlib

# Collect ALL test images and features — used by multiple tasks
print("Extracting features from test set...")

all_images   = []   # raw tensors (for display)
all_labels   = []   # ground truth class indices
all_features = []   # feature vectors (before final fc layer)
all_preds    = []   # predicted class indices

# We need features BEFORE the fc layer
# Use a hook to capture the output of the avgpool layer
# (the layer right before fc in ResNet18)
features_hook = []

def hook_fn(module, input, output):
    # output shape: (batch, 512, 1, 1) → flatten to (batch, 512)
    features_hook.append(output.squeeze(-1).squeeze(-1).detach())

# Register hook on the avgpool layer
handle = model.avgpool.register_forward_hook(hook_fn)

with torch.no_grad():
    for images, labels in testloader:
        _ = model(images)                    # forward pass triggers hook
        all_images.append(images)
        all_labels.append(labels)
        all_preds.append(torch.argmax(model(images), dim=1))

handle.remove()  # always remove hook when done!

all_images   = torch.cat(all_images,   dim=0)   # (N, 3, H, W)
all_labels   = torch.cat(all_labels,   dim=0)   # (N,)
all_preds    = torch.cat(all_preds,    dim=0)   # (N,)
all_features = torch.cat(features_hook[:len(all_images)//32 + 1], dim=0)
all_features = all_features[:len(all_images)]   # (N, 512)

print(f"Total test images: {len(all_images)}")
print(f"Feature shape per image: {all_features.shape[1]}")

In [ ]:
# NEAREST NEIGHBOURS IN FEATURE SPACE


def find_nearest_neighbours(query_idx, feature_matrix, k=5):
    """
    For a query image, find the k most similar images using cosine similarity.
    
    Args:
        query_idx: index of the query image in the dataset
        feature_matrix: (N, 512) tensor with all feature vectors
        k: number of neighbours to return
    
    Returns:
        indices of top-k neighbours (excluding the query itself)
    """
    query_vec = feature_matrix[query_idx].unsqueeze(0)  # (1, 512)

    # Normalize both vectors to unit length before dot product
    # This gives cosine similarity instead of dot product
    query_norm = F.normalize(query_vec,      dim=1)  # (1, 512)
    all_norm   = F.normalize(feature_matrix, dim=1)  # (N, 512)

    # Cosine similarity: (1, 512) × (512, N) = (1, N)
    similarities = torch.mm(query_norm, all_norm.T).squeeze(0)  # (N,)

    # Sort by similarity (highest first), exclude the query image itself
    sorted_indices = torch.argsort(similarities, descending=True)
    neighbours     = [i.item() for i in sorted_indices if i.item() != query_idx]

    return neighbours[:k], similarities

def visualize_nearest_neighbours(query_idx, k=5):
    """
    Display the query image and its k nearest neighbours side by side.
    """
    neighbour_indices, similarities = find_nearest_neighbours(
        query_idx, all_features, k=k)

    fig, axes = plt.subplots(1, k + 1, figsize=(3 * (k + 1), 4))

    # ── Display query image ───────────────────────────────────────────────
    query_img   = denormalize(all_images[query_idx])
    query_label = all_labels[query_idx].item()
    query_pred  = all_preds[query_idx].item()

    axes[0].imshow(query_img)
    axes[0].set_title(f"QUERY\nTrue: {query_label}\nPred: {query_pred}",
                      fontsize=9, color='blue', fontweight='bold')
    axes[0].axis('off')

    # ── Display each neighbour ────────────────────────────────────────────
    for plot_i, nb_idx in enumerate(neighbour_indices):
        nb_img   = denormalize(all_images[nb_idx])
        nb_label = all_labels[nb_idx].item()
        sim_val  = similarities[nb_idx].item()

        # Green title = same class as query, Red = different class
        color = 'green' if nb_label == query_label else 'red'

        axes[plot_i + 1].imshow(nb_img)
        axes[plot_i + 1].set_title(
            f"Rank {plot_i+1}\nLabel: {nb_label}\nSim: {sim_val:.3f}",
            fontsize=9, color=color)
        axes[plot_i + 1].axis('off')

    plt.suptitle(
        "Nearest Neighbours in Feature Space\n"
        "(Green = same class, Red = different class)",
        fontsize=11)
    plt.tight_layout()
    plt.show()

# Run for a few different query images
for query_idx in [0, 50, 100, 200]:
    visualize_nearest_neighbours(query_idx, k=5)


In [ ]:
# OCCLUSION SENSITIVITY HEATMAPS

def compute_occlusion_heatmap(image_tensor, model, target_class,
                               patch_size=16, stride=8):
    """
    Slide a black patch over the image and measure the score drop at each position.
    
    Args:
        image_tensor: (3, H, W) normalized image tensor
        model: trained PyTorch model
        target_class: the class index we're measuring the score for
        patch_size: size of the black square (e.g. 16 = 16×16 pixels)
        stride: how many pixels to move the patch each step
    
    Returns:
        heatmap: 2D numpy array with the score drops
    """
    model.eval()
    H, W = image_tensor.shape[1], image_tensor.shape[2]

    # Step 1: Get baseline score on the ORIGINAL (unoccluded) image
    with torch.no_grad():
        original_output = model(image_tensor.unsqueeze(0))  # (1, num_classes)
        # Use raw logit (before softmax) as the score
        baseline_score = original_output[0, target_class].item()

    # Step 2: Compute how many patch positions we'll have
    num_y = (H - patch_size) // stride + 1
    num_x = (W - patch_size) // stride + 1
    heatmap = np.zeros((num_y, num_x))

    # Step 3: Slide the patch over the image
    for row, y in enumerate(range(0, H - patch_size + 1, stride)):
        for col, x in enumerate(range(0, W - patch_size + 1, stride)):

            # Create a COPY of the image (don't modify the original)
            occluded = image_tensor.clone()

            # Set the patch region to zero (= black after normalization)
            # Zero in normalized space ≈ grey/black in pixel space
            occluded[:, y:y+patch_size, x:x+patch_size] = 0.0

            # Forward pass on the occluded image
            with torch.no_grad():
                output        = model(occluded.unsqueeze(0))
                occluded_score = output[0, target_class].item()

            # Score drop = how much covering this region hurt the model
            heatmap[row, col] = baseline_score - occluded_score

    return heatmap

def visualize_occlusion(img_idx, patch_size=16, stride=8):
    """
    Compute and display the occlusion heatmap for one image.
    """
    image_tensor = all_images[img_idx]
    true_label   = all_labels[img_idx].item()
    pred_label   = all_preds[img_idx].item()

    print(f"Computing occlusion heatmap... (this takes ~30 seconds)")

    # Compute heatmap for the PREDICTED class
    heatmap = compute_occlusion_heatmap(
        image_tensor, model,
        target_class=pred_label,
        patch_size=patch_size,
        stride=stride)

    # Upsample heatmap to original image size using bilinear interpolation
    H, W = image_tensor.shape[1], image_tensor.shape[2]
    heatmap_tensor = torch.tensor(heatmap).unsqueeze(0).unsqueeze(0)  # (1,1,h,w)
    heatmap_resized = F.interpolate(
        heatmap_tensor, size=(H, W), mode='bilinear', align_corners=False)
    heatmap_np = heatmap_resized.squeeze().numpy()

    # Normalize heatmap to [0, 1] for display
    heatmap_np = (heatmap_np - heatmap_np.min()) / \
                 (heatmap_np.max() - heatmap_np.min() + 1e-8)

    # Display: original image + heatmap + overlay
    original_img = denormalize(image_tensor)
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))

    axes[0].imshow(original_img)
    axes[0].set_title(f"Original Image\nTrue: {true_label} | Pred: {pred_label}",
                      fontsize=10)
    axes[0].axis('off')

    axes[1].imshow(heatmap_np, cmap='hot')
    axes[1].set_title("Occlusion Heatmap\n(brighter = more important)", fontsize=10)
    axes[1].axis('off')

    # Overlay: blend heatmap on top of original image
    # alpha=0.5 means 50% transparent
    axes[2].imshow(original_img)
    axes[2].imshow(heatmap_np, cmap='hot', alpha=0.5)
    axes[2].set_title("Overlay", fontsize=10)
    axes[2].axis('off')

    correct_str = "✓ CORRECT" if true_label == pred_label else "✗ WRONG"
    plt.suptitle(f"Occlusion Sensitivity — {correct_str}", fontsize=12)
    plt.tight_layout()
    plt.show()

# Find one correct and one incorrect prediction for comparison
correct_indices   = (all_preds == all_labels).nonzero(as_tuple=True)[0]
incorrect_indices = (all_preds != all_labels).nonzero(as_tuple=True)[0]

print("=== Correct prediction ===")
visualize_occlusion(correct_indices[0].item(), patch_size=16, stride=8)

print("=== Incorrect prediction ===")
visualize_occlusion(incorrect_indices[0].item(), patch_size=16, stride=8)

# ── Ce observi ────────────────────────────────────────────────────────────────
# Predicție CORECTĂ: heatmap-ul ar trebui să fie cald pe animalul însuși
#   → rețeaua se uită unde trebuie
# Predicție GREȘITĂ: heatmap-ul poate fi cald pe FUNDAL (iarbă, perete)
#   → rețeaua a "trișat" și a învățat caracteristici de fundal, nu ale animalului

In [ ]:
# TOP ACTIVATING PATCHES

# Storage for activation maps captured via hook
activation_maps = []

def make_activation_hook(storage):
    """
    Returns a hook function that saves the output of a layer.
    We use a factory function so we can attach multiple hooks.
    """
    def hook(module, input, output):
        # output shape: (batch, C, H, W)
        storage.append(output.detach())
    return hook

def get_top_activating_patches(model, dataloader, images_tensor,
                                layer_name='layer4', channel_k=0,
                                top_n=25, patch_size=32):
    """
    Find the top-N patches that maximally activate channel k in the given layer.
    
    Args:
        layer_name: which ResNet layer to hook ('layer1','layer2','layer3','layer4')
        channel_k:  which channel/filter to analyze
        top_n:      how many top patches to collect
        patch_size: size of the extracted patch in input image pixels
    
    Returns:
        list of (activation_value, image_index, patch as numpy array)
    """
    activation_maps.clear()

    # Attach hook to the chosen layer
    target_layer = getattr(model, layer_name)
    hook_handle   = target_layer.register_forward_hook(
                        make_activation_hook(activation_maps))

    all_activations = []  # list of (score, img_idx, y_feat, x_feat)

    with torch.no_grad():
        for batch_start in range(0, len(images_tensor), 32):
            batch = images_tensor[batch_start:batch_start+32]
            _     = model(batch)  # forward pass → triggers hook

    hook_handle.remove()

    # Process collected activation maps
    # Each element in activation_maps has shape (batch, C, H_feat, W_feat)
    img_idx = 0
    for batch_acts in activation_maps:
        batch_size_actual = batch_acts.shape[0]
        H_feat = batch_acts.shape[2]
        W_feat = batch_acts.shape[3]
        H_in   = images_tensor.shape[2]   # 224
        W_in   = images_tensor.shape[3]   # 224

        # Scale factor: how many input pixels per feature map cell
        scale_y = H_in / H_feat
        scale_x = W_in / W_feat

        for b in range(batch_size_actual):
            # Get the activation map for channel k, image b
            act_map = batch_acts[b, channel_k]  # (H_feat, W_feat)

            # Record every (activation_value, global_image_idx, feature_coords)
            for fy in range(H_feat):
                for fx in range(W_feat):
                    score = act_map[fy, fx].item()
                    all_activations.append((score, img_idx + b, fy, fx,
                                            scale_y, scale_x))
        img_idx += batch_size_actual

    # Sort by activation value, highest first
    all_activations.sort(key=lambda x: x[0], reverse=True)

    # Extract the actual image patches for the top-N activations
    top_patches = []
    for score, im_idx, fy, fx, scale_y, scale_x in all_activations[:top_n]:

        # Map feature map position → center pixel in original image
        center_y = int((fy + 0.5) * scale_y)
        center_x = int((fx + 0.5) * scale_x)

        # Extract patch centered at (center_y, center_x)
        half    = patch_size // 2
        y_start = max(0, center_y - half)
        y_end   = min(H_in, center_y + half)
        x_start = max(0, center_x - half)
        x_end   = min(W_in, center_x + half)

        patch_tensor = images_tensor[im_idx, :, y_start:y_end, x_start:x_end]
        patch_np     = denormalize(patch_tensor)

        top_patches.append((score, im_idx, patch_np))

    return top_patches

def visualize_top_patches(layer_name='layer4', channel_k=0, top_n=25):
    """
    Display a grid of the top-25 patches that maximally activate channel k.
    """
    print(f"Collecting top activations for {layer_name}, channel {channel_k}...")

    top_patches = get_top_activating_patches(
        model, testloader, all_images,
        layer_name=layer_name,
        channel_k=channel_k,
        top_n=top_n)

    # Display as a 5×5 grid
    grid_size = int(np.ceil(np.sqrt(top_n)))
    fig, axes = plt.subplots(grid_size, grid_size,
                              figsize=(grid_size * 2, grid_size * 2))

    for i, ax in enumerate(axes.flat):
        if i < len(top_patches):
            score, im_idx, patch = top_patches[i]
            ax.imshow(patch)
            ax.set_title(f"#{i+1}\nact={score:.2f}", fontsize=7)
        ax.axis('off')

    plt.suptitle(
        f"Top-{top_n} Patches for {layer_name}, Channel {channel_k}\n"
        f"(All patches that maximally activate this neuron)",
        fontsize=11)
    plt.tight_layout()
    plt.show()

# Analyze different layers and channels
# layer4 = high-level features (shapes, parts of animals)
# layer1 = low-level features (edges, colors)
visualize_top_patches(layer_name='layer4', channel_k=0,   top_n=25)
visualize_top_patches(layer_name='layer4', channel_k=100, top_n=25)
visualize_top_patches(layer_name='layer1', channel_k=5,   top_n=25)

# ── Ce observi ────────────────────────────────────────────────────────────────
# layer4, channel k → patch-uri cu aceeași PARTE DE ANIMAL (ex: ochi, blană, urechi)
#   Neuronul a învățat să detecteze un concept specific.
# layer1, channel k → patch-uri cu același TIP DE MUCHIE sau CULOARE
#   Neuronii timpurii detectează caracteristici simple.